## 1. Setup and Configuration

In [1]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta

In [2]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../../.env')

# Validate credentials
required_vars = ['DATABRICKS_HOST', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

✓ Environment configured


## 2. Initiative Parameters

### Initiative List
| Initiative | Brand | Pre Period | Repeat Period |
|------------|-------|------------|---------------|
| Srixon | Ariel Gel Ball | 4/13-5/13/2024 | →7/12/2024 |
| Srixon Boost | Ariel Gel Ball | 9/7-10/7/2024 | →12/6/2024 |
| Yoda | Ariel Gel Ball | 2/17-3/19/2025 | →5/18/2025 |
| Anakin | Ariel Gel Ball | 11/1-12/1/2025 | →12/30/2025 |
| Anakin (Fresh) | Ariel Gel Ball | 11/1-12/1/2025 | →12/30/2025 |
| Anakin (Clean) | Ariel Gel Ball | 11/1-12/1/2025 | →12/30/2025 |
| Anakin (BIO) | Ariel Gel Ball | 11/1-12/1/2025 | →12/30/2025 |
| Rapunzel (All) | Bold Gel Ball | 10/1-10/31/2025 | →12/30/2025 |
| Rapunzel Pink | Bold Gel Ball | 10/1-10/31/2025 | →12/30/2025 |
| Rapunzel Blue | Bold Gel Ball | 10/1-10/31/2025 | →12/30/2025 |
| Rapunzel WH-TEA&FL | Bold Gel Ball | 10/1-10/31/2025 | →12/30/2025 |
| Cinderella | Bold Gel Ball | 10/1-10/31/2024 | →12/30/2024 |
| Moana | Bold Gel Ball | 2/1-3/2/2024 | →5/1/2024 |
| Snowwhite | Bold Gel Ball | 4/12-5/12/2025 | →7/11/2025 |

### ⚡ Optimization Note
This notebook uses **consolidated data extraction** (2 queries instead of 33+) for faster runtime.

In [23]:
# Define initiative configurations
# Note: Sub-brand uses half-width katakana (半角カナ)
# ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ = Ariel Gel Ball
# ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ = Bold Gel Ball

initiatives = [
    # Ariel Gel Ball Initiatives
    {
        'initiative_name': 'Srixon',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,  # None = all variants
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-04-13',
        'pre_end': '2024-05-13',
        'repeat_start': '2024-04-13',
        'repeat_end': '2024-07-12'
    },
    {
        'initiative_name': 'Srixon Boost',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-09-07',
        'pre_end': '2024-10-07',
        'repeat_start': '2024-09-07',
        'repeat_end': '2024-12-06'
    },
    {
        'initiative_name': 'Yoda',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-02-17',
        'pre_end': '2025-03-19',
        'repeat_start': '2025-02-17',
        'repeat_end': '2025-05-18'
    },
    # Anakin - Latest Ariel Initiative with Variant Breakdown (Top 3)
    {
        'initiative_name': 'Anakin (All)',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-11-01',
        'pre_end': '2025-12-01',
        'repeat_start': '2025-11-01',
        'repeat_end': '2025-12-30'
    },
        {
        'initiative_name': 'Anakin (Blue)',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'Indoor Dry',
        'variant_filter': 'Ariel Gel Ball',  # jp_prod_family_1_name exact match
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-11-01',
        'pre_end': '2025-12-01',
        'repeat_start': '2025-11-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Anakin (Indoor Dry)',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'Indoor Dry',
        'variant_filter': 'Ariel Gel Ball_Indoor Dry',  # jp_prod_family_1_name exact match
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-11-01',
        'pre_end': '2025-12-01',
        'repeat_start': '2025-11-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Anakin (Pro Power)',
        'EN_sub_brand': 'Ariel Gel Ball',
        'sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'Pro Power',
        'variant_filter': 'Ariel Gel Ball_Pro Power',  # jp_prod_family_1_name exact match
        'repeat_sub_brand': 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-11-01',
        'pre_end': '2025-12-01',
        'repeat_start': '2025-11-01',
        'repeat_end': '2025-12-30'
    },
    # Bold Gel Ball Initiatives
    {
        'initiative_name': 'Rapunzel (All)',
        'EN_sub_brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
        'repeat_start': '2025-10-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Rapunzel (Pink)',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'pink',
        'variant_filter': 'Bold Gel Ball_Pink',  # jp_prod_family_1_name exact match
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
        'repeat_start': '2025-10-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Rapunzel (Blue)',
        'EN_sub_brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'Blue',
        'variant_filter': 'Bold Gel Ball_Blue',  # jp_prod_family_1_name exact match
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
        'repeat_start': '2025-10-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Rapunzel (WH-TEA&FL)',
        'brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'WH-TEA&FL',
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-10-01',
        'pre_end': '2025-10-31',
        'repeat_start': '2025-10-01',
        'repeat_end': '2025-12-30'
    },
    {
        'initiative_name': 'Cinderella',
        'EN_sub_brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-10-01',
        'pre_end': '2024-10-31',
        'repeat_start': '2024-10-01',
        'repeat_end': '2024-12-30'
    },
    {
        'initiative_name': 'Moana',
        'EN_sub_brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2024-02-01',
        'pre_end': '2024-03-02',
        'repeat_start': '2024-02-01',
        'repeat_end': '2024-05-01'
    },
    {
        'initiative_name': 'Snowwhite',
        'EN_sub_brand': 'Bold Gel Ball',
        'sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'variant': 'all',
        'variant_filter': None,
        'repeat_sub_brand': 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ',
        'pre_start': '2025-04-12',
        'pre_end': '2025-05-12',
        'repeat_start': '2025-04-12',
        'repeat_end': '2025-07-11'
    }
]

# Customer filters (all IDPOS retailers, excluding CVS)
customer_codes = ['cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009', 
                  'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013']
customer_filter_sql = "', '".join(customer_codes)

# Category
category = 'Laundry'

# Convert initiatives to DataFrame for consolidated query
initiatives_df = pd.DataFrame(initiatives)

print(f"✓ Parameters configured")
print(f"  Initiatives: {len(initiatives)}")
print(f"  - Ariel Gel Ball: {len([i for i in initiatives if i['EN_sub_brand'] == 'Ariel Gel Ball'])}")
print(f"  - Bold Gel Ball: {len([i for i in initiatives if i['EN_sub_brand'] == 'Bold Gel Ball'])}")
print(f"  Customers: {len(customer_codes)}")
print(f"  Category: {category}")

✓ Parameters configured
  Initiatives: 13
  - Ariel Gel Ball: 6
  - Bold Gel Ball: 7
  Customers: 9
  Category: Laundry


In [ ]:
# =============================================================================
# OPTIMIZED: Consolidated Data Extraction Functions
# Instead of 33 separate queries, we extract all data in 2 brand-level queries
# =============================================================================

def get_db_connection():
    """Create a fresh Databricks connection with retry logic."""
    return sql.connect(
        server_hostname=os.getenv("DATABRICKS_HOST"),
        http_path=os.getenv("DATABRICKS_HTTP_PATH"),
        access_token=os.getenv("DATABRICKS_TOKEN")
    )

def build_initiatives_cte(initiatives_list, brand_filter=None):
    """
    Build SQL VALUES clause for initiative metadata.
    This allows us to join initiative parameters directly in SQL.
    """
    filtered = [i for i in initiatives_list if brand_filter is None or i['brand'] == brand_filter]
    
    values_rows = []
    for init in filtered:
        variant_like = init['variant_filter'] if init['variant_filter'] else ''
        values_rows.append(
            f"('{init['initiative_name']}', '{init['sub_brand']}', '{init['repeat_sub_brand']}', "
            f"'{variant_like}', '{init['pre_start']}', '{init['pre_end']}', '{init['repeat_end']}')"
        )
    
    return f"""
    SELECT * FROM (VALUES
        {','.join(values_rows)}
    ) AS t(initiative_name, sub_brand, repeat_sub_brand, variant_like, pre_start, pre_end, repeat_end)
    """

def extract_brand_data(connection, initiatives_list, brand_name, sub_brand_code, max_retries=3):
    """
    Extract all trial and repeat transaction data for a brand in ONE query.
    Also extracts next purchase data for non-repeaters (Anakin/Rapunzel only).
    
    Returns TWO DataFrames:
    1. Main transactions: shopper_key, initiative_name, trial_date, repeat_date, pack_size, has_repeat
    2. Next purchases: shopper_key, initiative_name, trial_date, next_sub_brand, next_date, days_to_next
    """
    
    # Filter initiatives for this brand
    brand_initiatives = [i for i in initiatives_list if i['brand'] == brand_name]
    if not brand_initiatives:
        return pd.DataFrame()
    
    # Calculate overall date boundaries for this brand
    min_date = min(i['pre_start'] for i in brand_initiatives)
    max_date = max(i['repeat_end'] for i in brand_initiatives)
    
    # Build initiative metadata CTE
    init_cte = build_initiatives_cte(brand_initiatives)
    
    query = f"""
    WITH initiatives AS (
        {init_cte}
    ),
    -- Step 1: Get ALL transactions for this brand within the overall date window
    brand_transactions AS (
        SELECT
            idpos.shopper_key,
            sales_period_group_end_date_part AS txn_date,
            prod.jp_prod_family_1_name AS variant_name,
            jp_segment_4_name AS pack_size,
            jp_sub_brand_alter_lang_name AS sub_brand
        FROM
            cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
            LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
            LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
        WHERE
            jp_category_name = '{category}'
            AND jp_sub_brand_alter_lang_name = '{sub_brand_code}'
            AND idpos.data_provider_code_part IN ('{customer_filter_sql}')
            AND sales_period_group_end_date_part BETWEEN '{min_date}' AND '{max_date}'
            AND shopper.member_ind = 'Y'
    ),
    -- Step 2: Identify trial shoppers per initiative (with variant filter applied using jp_prod_family_1_name)
    trial_shoppers AS (
        SELECT
            bt.shopper_key,
            i.initiative_name,
            MIN(bt.txn_date) AS trial_date,
            i.repeat_end
        FROM brand_transactions bt
        CROSS JOIN initiatives i
        WHERE
            bt.txn_date BETWEEN i.pre_start AND i.pre_end
            AND (i.variant_like = '' OR bt.variant_name = i.variant_like)
        GROUP BY bt.shopper_key, i.initiative_name, i.repeat_end
    ),
    -- Step 3: Find repeat transactions (after trial_date, before repeat_end, any variant in sub-brand)
    repeat_transactions AS (
        SELECT
            ts.shopper_key,
            ts.initiative_name,
            ts.trial_date,
            bt.txn_date AS repeat_date,
            bt.pack_size,
            FLOOR(DATEDIFF(bt.txn_date, ts.trial_date) / 7) + 1 AS week_number
        FROM trial_shoppers ts
        INNER JOIN brand_transactions bt ON ts.shopper_key = bt.shopper_key
        WHERE bt.txn_date > ts.trial_date
          AND bt.txn_date <= ts.repeat_end
    )
    -- Final output: combine trial counts and repeat data
    SELECT
        ts.initiative_name,
        ts.shopper_key,
        ts.trial_date,
        rt.repeat_date,
        rt.pack_size,
        rt.week_number,
        CASE WHEN rt.repeat_date IS NOT NULL THEN 1 ELSE 0 END AS has_repeat
    FROM trial_shoppers ts
    LEFT JOIN repeat_transactions rt 
        ON ts.shopper_key = rt.shopper_key 
        AND ts.initiative_name = rt.initiative_name
    """
    
    # Separate query for next purchase analysis (Anakin/Rapunzel only)
    # Following 03_next_purchase template logic
            # Extract main transaction data
            with connection.cursor() as cursor:
                cursor.execute(query)
                result = cursor.fetchall()
                columns = [desc[0] for desc in cursor.description]
                df = pd.DataFrame(result, columns=columns)
                print(f"  ✓ Extracted {len(df):,} transaction rows for {brand_name}")
            
            # Extract next purchase data (only for Anakin/Rapunzel)
            next_df = pd.DataFrame()
            if any(init['initiative_name'] in ['Anakin (All)', 'Rapunzel (All)'] 
                   for init in brand_initiatives):
                with connection.cursor() as cursor:
                    cursor.execute(next_purchase_query)
                    result = cursor.fetchall()
                    columns = [desc[0] for desc in cursor.description]
                    next_df = pd.DataFrame(result, columns=columns)
                    print(f"  ✓ Extracted {len(next_df):,} next purchase rows for {brand_name}")
            
            return df, next_df
            
        except Exception as e:
            print(f"  ⚠ Attempt {attempt + 1}/{max_retries} failed: {e}")
            if attempt < max_retries - 1:
            act.shopper_key,print("✓ Consolidated data extraction functions defined")
                print("    Reconnecting...")
            i.initiative_name,

                try:
            MIN(act.txn_date) AS trial_date,    return pd.DataFrame()

                    connection.close()
            i.repeat_end,    

                except:
            i.sub_brand AS trial_sub_brand                return pd.DataFrame()

                    pass
        FROM all_category_txns act                print(f"  ✗ All retries exhausted for {brand_name}")

                connection = get_db_connection()
        CROSS JOIN initiatives i            else:

            else:
        WHERE                connection = get_db_connection()

                print(f"  ✗ All retries exhausted for {brand_name}")
            act.txn_date BETWEEN i.pre_start AND i.pre_end                    pass

                return pd.DataFrame(), pd.DataFrame()
            AND act.sub_brand = i.sub_brand                except:

    
        GROUP BY act.shopper_key, i.initiative_name, i.repeat_end, i.sub_brand                    connection.close()

    return pd.DataFrame(), pd.DataFrame()
    ),                try:


    -- Find FIRST purchase after trial (following 03_next_purchase logic)                print("    Reconnecting...")

print("✓ Consolidated data extraction functions defined")
    next_purchases_with_seq AS (            if attempt < max_retries - 1:

        SELECT            print(f"  ⚠ Attempt {attempt + 1}/{max_retries} failed: {e}")

            ts.shopper_key,        except Exception as e:

            ts.initiative_name,                return df

            ts.trial_date,                print(f"  ✓ Extracted {len(df):,} rows for {brand_name}")

            ts.trial_sub_brand,                df = pd.DataFrame(result, columns=columns)

            act.txn_date AS next_date,                columns = [desc[0] for desc in cursor.description]

            act.sub_brand AS next_sub_brand,                result = cursor.fetchall()

            DATEDIFF(act.txn_date, ts.trial_date) AS days_to_next,                cursor.execute(query)

            ROW_NUMBER() OVER (            with connection.cursor() as cursor:

                PARTITION BY ts.shopper_key, ts.initiative_name        try:

                ORDER BY act.txn_date    for attempt in range(max_retries):

            ) AS purchase_sequence    

        FROM trial_shoppers ts    """

        INNER JOIN all_category_txns act ON ts.shopper_key = act.shopper_key      AND initiative_name IN ('Anakin (All)', 'Rapunzel (All)')

        WHERE act.txn_date > ts.trial_date    WHERE purchase_sequence = 1

          AND act.txn_date <= ts.repeat_end    FROM next_purchases_with_seq

    )        days_to_next

    SELECT        next_sub_brand,

        shopper_key,        next_date,

        initiative_name,        trial_sub_brand,
        trial_date,

✓ Consolidated data extraction functions defined


In [5]:
# =============================================================================
# Metric Calculation Functions (operate on cached DataFrames, no DB calls)
# =============================================================================

def calculate_overall_metrics(df, initiatives_list):
    """
    Calculate overall repeat rates from cached transaction data.
    No database calls - pure pandas operations.
    """
    results = []
    
    for init in initiatives_list:
        init_name = init['initiative_name']
        init_data = df[df['initiative_name'] == init_name]
        
        if len(init_data) == 0:
            continue
        
        # Count unique trial shoppers (all rows have trial info)
        pre_shoppers = init_data['shopper_key'].nunique()
        
        # Count unique repeat shoppers (those with at least one repeat_date)
        repeat_data = init_data[init_data['repeat_date'].notna()]
        repeat_shoppers = repeat_data['shopper_key'].nunique()
        
        repeat_rate = (repeat_shoppers / pre_shoppers * 100) if pre_shoppers > 0 else 0
        
        results.append({
            'initiative_name': init_name,
            'brand': init['brand'],
            'variant': init['variant'],
            'pre_period': f"{init['pre_start']} to {init['pre_end']}",
            'repeat_period': f"{init['repeat_start']} to {init['repeat_end']}",
            'pre_shoppers': pre_shoppers,
            'repeat_shoppers': repeat_shoppers,
            'repeat_rate': repeat_rate
        })
    
    return pd.DataFrame(results)

def calculate_weekly_metrics(df, initiatives_list):
    """
    Calculate weekly repeat rate dynamics from cached data.
    """
    all_weekly = []
    
    for init in initiatives_list:
        init_name = init['initiative_name']
        init_data = df[df['initiative_name'] == init_name]
        
        if len(init_data) == 0:
            continue
        
        pre_shoppers = init_data['shopper_key'].nunique()
        
        # Get repeat data with week numbers
        repeat_data = init_data[init_data['repeat_date'].notna()].copy()
        
        if len(repeat_data) == 0:
            continue
        
        # Find first repeat week per shopper
        first_repeat = repeat_data.groupby('shopper_key')['week_number'].min().reset_index()
        first_repeat.columns = ['shopper_key', 'first_repeat_week']
        
        # Count new repeaters per week
        weekly_new = first_repeat.groupby('first_repeat_week').size().reset_index(name='new_repeaters')
        weekly_new.columns = ['week', 'new_repeaters']
        weekly_new = weekly_new.sort_values('week')
        
        # Calculate cumulative metrics
        weekly_new['cumulative_repeaters'] = weekly_new['new_repeaters'].cumsum()
        weekly_new['total_pre_shoppers'] = pre_shoppers
        weekly_new['cumulative_repeat_rate'] = (weekly_new['cumulative_repeaters'] / pre_shoppers * 100).round(2)
        weekly_new['incremental_repeat_rate'] = (weekly_new['new_repeaters'] / pre_shoppers * 100).round(2)
        weekly_new['initiative_name'] = init_name
        
        all_weekly.append(weekly_new)
    
    return pd.concat(all_weekly, ignore_index=True) if all_weekly else pd.DataFrame()

def calculate_size_metrics(df, initiatives_list):
    """
    Calculate size migration metrics from cached data.
    """
    all_size = []
    
    for init in initiatives_list:
        init_name = init['initiative_name']
        init_data = df[df['initiative_name'] == init_name]
        
        if len(init_data) == 0:
            continue
        
        pre_shoppers = init_data['shopper_key'].nunique()
        
        # Get repeat data with pack sizes
        repeat_data = init_data[init_data['repeat_date'].notna()].copy()
        
        if len(repeat_data) == 0:
            continue
        
        # Count unique shoppers per size
        size_breakdown = repeat_data.groupby('pack_size')['shopper_key'].nunique().reset_index()
        size_breakdown.columns = ['repeat_size', 'repeat_shoppers']
        size_breakdown = size_breakdown.sort_values('repeat_shoppers', ascending=False)
        
        size_breakdown['total_pre_shoppers'] = pre_shoppers
        size_breakdown['repeat_rate_to_size'] = (size_breakdown['repeat_shoppers'] / pre_shoppers * 100).round(2)
        size_breakdown['share_of_repeaters'] = (size_breakdown['repeat_shoppers'] / size_breakdown['repeat_shoppers'].sum() * 100).round(2)
        size_breakdown['initiative_name'] = init_name
        
        all_size.append(size_breakdown)
    
    return pd.concat(all_size, ignore_index=True) if all_size else pd.DataFrame()

print("✓ Metric calculation functions defined (pandas-based, no DB calls)")

✓ Metric calculation functions defined (pandas-based, no DB calls)


In [6]:
# =============================================================================
# STEP 3: Consolidated Data Extraction (2 queries instead of 33+)
# =============================================================================

print("=" * 70)
print("EXTRACTING DATA - CONSOLIDATED APPROACH")
print("=" * 70)
print("\nThis optimized approach extracts all data in just 2 queries:")
print("  • Query 1: All Ariel Gel Ball initiatives")
print("  • Query 2: All Bold Gel Ball initiatives")
print("\nPrevious approach: 33 separate queries (11 initiatives × 3 analyses)")
print("-" * 70)

import time
start_time = time.time()

# Initialize storage
ariel_df = pd.DataFrame()
bold_df = pd.DataFrame()

# Extract Ariel Gel Ball data
print("\n📊 Extracting Ariel Gel Ball data...")
try:
    with get_db_connection() as conn:
        ariel_df = extract_brand_data(
            conn, 
            initiatives, 
            brand_name='Ariel Gel Ball',
            sub_brand_code='ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'
        )
except Exception as e:
    print(f"  ✗ Ariel extraction failed: {e}")

# Extract Bold Gel Ball data  
print("\n📊 Extracting Bold Gel Ball data...")
try:
    with get_db_connection() as conn:
        bold_df = extract_brand_data(
            conn,
            initiatives,
            brand_name='Bold Gel Ball', 
            sub_brand_code='ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ'
        )
except Exception as e:
    print(f"  ✗ Bold extraction failed: {e}")

# Combine all data
all_txn_df = pd.concat([ariel_df, bold_df], ignore_index=True)

extraction_time = time.time() - start_time
print("\n" + "=" * 70)
print(f"✓ Data extraction completed in {extraction_time:.1f} seconds")
print(f"  Total rows cached: {len(all_txn_df):,}")
print(f"  Unique initiatives: {all_txn_df['initiative_name'].nunique() if len(all_txn_df) > 0 else 0}")
print("=" * 70)

EXTRACTING DATA - CONSOLIDATED APPROACH

This optimized approach extracts all data in just 2 queries:
  • Query 1: All Ariel Gel Ball initiatives
  • Query 2: All Bold Gel Ball initiatives

Previous approach: 33 separate queries (11 initiatives × 3 analyses)
----------------------------------------------------------------------

📊 Extracting Ariel Gel Ball data...


HTTP request failed after retries: HTTPSConnectionPool(host='https', port=443): Max retries exceeded with url: //adb-2258763851730787.7.azuredatabricks.net/api/2.0/connector-service/feature-flags/PYTHON/4.2.3 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000295C4F31E10>: Failed to resolve 'https' ([Errno 11001] getaddrinfo failed)"))


  ✓ Extracted 1,869,617 rows for Ariel Gel Ball

📊 Extracting Bold Gel Ball data...


HTTP request error: 'NoneType' object has no attribute 'request'


  ✓ Extracted 2,054,322 rows for Bold Gel Ball

✓ Data extraction completed in 1208.0 seconds
  Total rows cached: 3,923,939
  Unique initiatives: 8


## 4. Calculate Metrics from Cached Data (No Additional DB Calls)

In [7]:
# =============================================================================
# Calculate ALL metrics from cached data (no more DB queries!)
# =============================================================================

print("Calculating metrics from cached data...")
print("-" * 50)

# Overall repeat rates
print("\n📊 Calculating overall repeat rates...")
overall_df = calculate_overall_metrics(all_txn_df, initiatives)
print(f"  ✓ Processed {len(overall_df)} initiatives")

# Weekly dynamics
print("\n📊 Calculating weekly dynamics...")
all_weekly_df = calculate_weekly_metrics(all_txn_df, initiatives)
print(f"  ✓ Generated {len(all_weekly_df)} weekly data points")

# Size migration
print("\n📊 Calculating size migration...")
all_size_df = calculate_size_metrics(all_txn_df, initiatives)
print(f"  ✓ Generated {len(all_size_df)} size breakdown rows")

# Brand switching analysis for non-repeaters
print("\n📊 Analyzing brand switching for non-repeaters...")
try:
    with get_db_connection() as conn:
        switching_df = analyze_brand_switching(conn, initiatives, all_txn_df)
        if len(switching_df) > 0:
            print(f"  ✓ Brand switching analysis completed")
        else:
            print("  ! No switching data available")
except Exception as e:
    print(f"  ✗ Brand switching analysis failed: {e}")
    switching_df = pd.DataFrame()

print("\n" + "=" * 50)
print("✓ All metrics calculated from cached data")
print("  (No additional database queries required)")
print("=" * 50)

Calculating metrics from cached data...
--------------------------------------------------

📊 Calculating overall repeat rates...
  ✓ Processed 8 initiatives

📊 Calculating weekly dynamics...
  ✓ Generated 100 weekly data points

📊 Calculating size migration...
  ✓ Generated 68 size breakdown rows

✓ All metrics calculated from cached data
  (No additional database queries required)


In [8]:
# =============================================================================
# Display Overall Results
# =============================================================================

print("\n" + "="*90)
print("INITIATIVE REPEAT RATE COMPARISON - OVERALL RESULTS")
print("="*90)
print(f"\nAnalysis Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"Channels: All IDPOS retailers (excluding CVS)")
print(f"Total Initiatives Analyzed: {len(overall_df)}")
print("\n")

if len(overall_df) > 0:
    # Format for display
    display_overall = overall_df.copy()
    display_overall['pre_shoppers'] = display_overall['pre_shoppers'].apply(lambda x: f"{x:,}")
    display_overall['repeat_shoppers'] = display_overall['repeat_shoppers'].apply(lambda x: f"{x:,}")
    display_overall['repeat_rate'] = display_overall['repeat_rate'].apply(lambda x: f"{x:.2f}%")
    
    display(display_overall[['initiative_name', 'brand', 'variant', 'pre_period', 'repeat_period', 
                             'pre_shoppers', 'repeat_shoppers', 'repeat_rate']])
else:
    print("⚠ No results to display. Check data extraction logs above.")


INITIATIVE REPEAT RATE COMPARISON - OVERALL RESULTS

Analysis Date: 2026-01-16
Channels: All IDPOS retailers (excluding CVS)
Total Initiatives Analyzed: 8




,initiative_name,brand,variant,pre_period,repeat_period,pre_shoppers,repeat_shoppers,repeat_rate
0,Srixon,Ariel Gel Ball,all,2024-04-13 to 2024-05-13,2024-04-13 to 2024-07-12,"277,787","114,782",41.32%
1,Srixon Boost,Ariel Gel Ball,all,2024-09-07 to 2024-10-07,2024-09-07 to 2024-12-06,"260,802","113,332",43.46%
2,Yoda,Ariel Gel Ball,all,2025-02-17 to 2025-03-19,2025-02-17 to 2025-05-18,"335,255","128,727",38.40%
3,Anakin (All),Ariel Gel Ball,all,2025-11-01 to 2025-12-01,2025-11-01 to 2025-12-30,"299,289","89,308",29.84%
4,Rapunzel (All),Bold Gel Ball,all,2025-10-01 to 2025-10-31,2025-10-01 to 2025-12-30,"341,522","134,987",39.53%
5,Cinderella,Bold Gel Ball,all,2024-10-01 to 2024-10-31,2024-10-01 to 2024-12-30,"259,020","110,067",42.49%
6,Moana,Bold Gel Ball,all,2024-02-01 to 2024-03-02,2024-02-01 to 2024-05-01,"294,581","119,172",40.45%
7,Snowwhite,Bold Gel Ball,all,2025-04-12 to 2025-05-12,2025-04-12 to 2025-07-11,"329,972","133,679",40.51%


## 5. Weekly Dynamics Analysis

In [9]:
# Display weekly dynamics comparison (already calculated from cached data)
if len(all_weekly_df) > 0:
    print("\n" + "="*90)
    print("WEEKLY REPEAT RATE DYNAMICS - CUMULATIVE REPEAT RATE BY WEEK")
    print("="*90)
    print("\nShows how repeat rate builds up over time:")
    print("  • Week 1-2: Initial momentum / early adopters")
    print("  • Week 3-8: Sustained engagement pattern")
    print("  • Week 8+: Long-tail repeat behavior")
    print("\n")
    
    # Pivot for easy comparison
    pivot_weekly = all_weekly_df.pivot_table(
        index='week',
        columns='initiative_name',
        values='cumulative_repeat_rate',
        aggfunc='first'
    ).round(2)
    
    display(pivot_weekly)
else:
    print("⚠ No weekly dynamics data available.")


WEEKLY REPEAT RATE DYNAMICS - CUMULATIVE REPEAT RATE BY WEEK

Shows how repeat rate builds up over time:
  • Week 1-2: Initial momentum / early adopters
  • Week 3-8: Sustained engagement pattern
  • Week 8+: Long-tail repeat behavior




initiative_name,Anakin (All),Cinderella,Moana,Rapunzel (All),Snowwhite,Srixon,Srixon Boost,Yoda
week,,,,,,,,
1.0,2.41,2.24,2.27,3.16,3.55,1.79,1.32,3.35
2.0,6.93,6.42,6.29,8.01,8.39,5.33,4.72,7.93
3.0,12.08,11.31,11.01,12.96,13.42,9.85,9.57,12.82
4.0,17.57,16.62,16.02,17.69,18.55,15.07,15.05,17.72
5.0,23.09,22.49,21.78,22.72,23.75,21.10,21.79,22.76
6.0,26.39,26.90,25.96,26.31,27.58,25.62,26.64,26.35
7.0,28.48,30.65,29.52,29.39,30.78,29.48,30.74,29.28
8.0,29.56,34.05,32.67,32.20,33.70,32.95,34.43,32.01
9.0,29.84,37.63,35.96,35.03,36.65,36.51,38.21,34.69


In [19]:
# =============================================================================
# VISUALIZATION 1: Ariel Gel Ball - All Initiatives Weekly Repeat Rate
# (Interactive with Anakin highlighted)
# =============================================================================
import plotly.graph_objects as go

# Filter Ariel Gel Ball initiatives
ariel_initiatives = ['Srixon', 'Srixon Boost', 'Yoda', 'Anakin (All)']
ariel_weekly = all_weekly_df[all_weekly_df['initiative_name'].isin(ariel_initiatives)]

if len(ariel_weekly) > 0:
    fig = go.Figure()
    
    # Style settings: Anakin highlighted, others grayed out
    for init in ariel_initiatives:
        data = ariel_weekly[ariel_weekly['initiative_name'] == init].sort_values('week')
        if len(data) > 0:
            is_highlight = 'Anakin' in init
            
            fig.add_trace(go.Scatter(
                x=data['week'],
                y=data['cumulative_repeat_rate'],
                mode='lines+markers',
                name=init,
                line=dict(
                    color='#d62728' if is_highlight else '#cccccc',
                    width=3 if is_highlight else 1.5
                ),
                marker=dict(
                    size=8 if is_highlight else 5,
                    color='#d62728' if is_highlight else '#cccccc'
                ),
                hovertemplate='<b>%{fullData.name}</b><br>Week: %{x}<br>Repeat Rate: %{y:.2f}%<extra></extra>'
            ))
    
    fig.update_layout(
        title='Ariel Gel Ball - Weekly Repeat Rate by Initiative',
        xaxis_title='Week',
        yaxis_title='Cumulative Repeat Rate (%)',
        hovermode='closest',
        height=500,
        showlegend=True,
        legend=dict(x=0.7, y=0.2)
    )
    
    fig.update_xaxes(range=[0, 14], gridcolor='lightgray')
    fig.update_yaxes(range=[0, 50], gridcolor='lightgray')
    
    fig.show()
    
    # Display reference table
    print("\n📊 Reference Table: Ariel Gel Ball Weekly Data")
    pivot_ariel = ariel_weekly.pivot_table(
        index='week',
        columns='initiative_name',
        values='cumulative_repeat_rate',
        aggfunc='first'
    ).round(2)
    display(pivot_ariel)
else:
    print("⚠ No Ariel weekly data available")


📊 Reference Table: Ariel Gel Ball Weekly Data


initiative_name,Anakin (All),Srixon,Srixon Boost,Yoda
week,,,,
1.0,2.41,1.79,1.32,3.35
2.0,6.93,5.33,4.72,7.93
3.0,12.08,9.85,9.57,12.82
4.0,17.57,15.07,15.05,17.72
5.0,23.09,21.10,21.79,22.76
6.0,26.39,25.62,26.64,26.35
7.0,28.48,29.48,30.74,29.28
8.0,29.56,32.95,34.43,32.01
9.0,29.84,36.51,38.21,34.69


In [20]:
# =============================================================================
# VISUALIZATION 2: Bold Gel Ball - All Initiatives Weekly Repeat Rate
# (Interactive with Rapunzel highlighted)
# =============================================================================

# Filter Bold Gel Ball initiatives
bold_initiatives = ['Rapunzel (All)', 'Cinderella', 'Moana', 'Snowwhite']
bold_weekly = all_weekly_df[all_weekly_df['initiative_name'].isin(bold_initiatives)]

if len(bold_weekly) > 0:
    fig = go.Figure()
    
    # Style settings: Rapunzel highlighted, others grayed out
    for init in bold_initiatives:
        data = bold_weekly[bold_weekly['initiative_name'] == init].sort_values('week')
        if len(data) > 0:
            is_highlight = 'Rapunzel' in init
            
            fig.add_trace(go.Scatter(
                x=data['week'],
                y=data['cumulative_repeat_rate'],
                mode='lines+markers',
                name=init,
                line=dict(
                    color='#9467bd' if is_highlight else '#cccccc',
                    width=3 if is_highlight else 1.5
                ),
                marker=dict(
                    size=8 if is_highlight else 5,
                    color='#9467bd' if is_highlight else '#cccccc'
                ),
                hovertemplate='<b>%{fullData.name}</b><br>Week: %{x}<br>Repeat Rate: %{y:.2f}%<extra></extra>'
            ))
    
    fig.update_layout(
        title='Bold Gel Ball - Weekly Repeat Rate by Initiative',
        xaxis_title='Week',
        yaxis_title='Cumulative Repeat Rate (%)',
        hovermode='closest',
        height=500,
        showlegend=True,
        legend=dict(x=0.7, y=0.2)
    )
    
    fig.update_xaxes(range=[0, 14], gridcolor='lightgray')
    fig.update_yaxes(range=[0, 50], gridcolor='lightgray')
    
    fig.show()
    
    # Display reference table
    print("\n📊 Reference Table: Bold Gel Ball Weekly Data")
    pivot_bold = bold_weekly.pivot_table(
        index='week',
        columns='initiative_name',
        values='cumulative_repeat_rate',
        aggfunc='first'
    ).round(2)
    display(pivot_bold)
else:
    print("⚠ No Bold weekly data available")


📊 Reference Table: Bold Gel Ball Weekly Data


initiative_name,Cinderella,Moana,Rapunzel (All),Snowwhite
week,,,,
1.0,2.24,2.27,3.16,3.55
2.0,6.42,6.29,8.01,8.39
3.0,11.31,11.01,12.96,13.42
4.0,16.62,16.02,17.69,18.55
5.0,22.49,21.78,22.72,23.75
6.0,26.90,25.96,26.31,27.58
7.0,30.65,29.52,29.39,30.78
8.0,34.05,32.67,32.20,33.70
9.0,37.63,35.96,35.03,36.65


In [21]:
# =============================================================================
# VISUALIZATION 3: Ariel Gel Ball Anakin - By Variant Weekly Repeat Rate
# (Interactive variant comparison)
# =============================================================================

# Filter Anakin variants
anakin_variants_list = ['Anakin (All)', 'Anakin (Fresh)', 'Anakin (Clean)', 'Anakin (BIO)']
anakin_weekly = all_weekly_df[all_weekly_df['initiative_name'].isin(anakin_variants_list)]

if len(anakin_weekly) > 0 and len(anakin_weekly['initiative_name'].unique()) > 1:
    fig = go.Figure()
    
    # Color palette for variants
    variant_colors = {
        'Anakin (All)': '#d62728',
        'Anakin (Fresh)': '#1f77b4',
        'Anakin (Clean)': '#2ca02c',
        'Anakin (BIO)': '#ff7f0e'
    }
    
    for variant in anakin_variants_list:
        data = anakin_weekly[anakin_weekly['initiative_name'] == variant].sort_values('week')
        if len(data) > 0:
            is_all = '(All)' in variant
            
            fig.add_trace(go.Scatter(
                x=data['week'],
                y=data['cumulative_repeat_rate'],
                mode='lines+markers',
                name=variant,
                line=dict(
                    color=variant_colors.get(variant, '#666666'),
                    width=3 if is_all else 2,
                    dash='solid' if is_all else 'dot'
                ),
                marker=dict(size=7),
                hovertemplate='<b>%{fullData.name}</b><br>Week: %{x}<br>Repeat Rate: %{y:.2f}%<extra></extra>'
            ))
    
    fig.update_layout(
        title='Ariel Gel Ball - Anakin Variant Comparison',
        xaxis_title='Week',
        yaxis_title='Cumulative Repeat Rate (%)',
        hovermode='closest',
        height=500,
        showlegend=True,
        legend=dict(x=0.7, y=0.2)
    )
    
    fig.update_xaxes(range=[0, 14], gridcolor='lightgray')
    fig.update_yaxes(range=[0, 50], gridcolor='lightgray')
    
    fig.show()
    
    # Display reference table
    print("\n📊 Reference Table: Anakin Variant Weekly Data")
    pivot_anakin = anakin_weekly.pivot_table(
        index='week',
        columns='initiative_name',
        values='cumulative_repeat_rate',
        aggfunc='first'
    ).round(2)
    display(pivot_anakin)
else:
    print("⚠ No Anakin variant weekly data available")
    print("   Note: Variant-level data requires correct product name filters.")
    print("   Currently showing only 'Anakin (All)' - need to re-extract with variant patterns.")

⚠ No Anakin variant weekly data available
   Note: Variant-level data requires correct product name filters.
   Currently showing only 'Anakin (All)' - need to re-extract with variant patterns.


In [22]:
# =============================================================================
# VISUALIZATION 4: Bold Gel Ball Rapunzel - By Variant Weekly Repeat Rate
# (Interactive variant comparison)
# =============================================================================

# Filter Rapunzel variants
rapunzel_variants_list = ['Rapunzel (All)', 'Rapunzel (Pink)', 'Rapunzel (Blue)', 'Rapunzel (WH-TEA&FL)']
rapunzel_weekly = all_weekly_df[all_weekly_df['initiative_name'].isin(rapunzel_variants_list)]

if len(rapunzel_weekly) > 0 and len(rapunzel_weekly['initiative_name'].unique()) > 1:
    fig = go.Figure()
    
    # Color palette for variants
    variant_colors = {
        'Rapunzel (All)': '#9467bd',
        'Rapunzel (Pink)': '#ff69b4',
        'Rapunzel (Blue)': '#4169e1',
        'Rapunzel (WH-TEA&FL)': '#dda0dd'
    }
    
    for variant in rapunzel_variants_list:
        data = rapunzel_weekly[rapunzel_weekly['initiative_name'] == variant].sort_values('week')
        if len(data) > 0:
            is_all = '(All)' in variant
            
            fig.add_trace(go.Scatter(
                x=data['week'],
                y=data['cumulative_repeat_rate'],
                mode='lines+markers',
                name=variant,
                line=dict(
                    color=variant_colors.get(variant, '#666666'),
                    width=3 if is_all else 2,
                    dash='solid' if is_all else 'dot'
                ),
                marker=dict(size=7),
                hovertemplate='<b>%{fullData.name}</b><br>Week: %{x}<br>Repeat Rate: %{y:.2f}%<extra></extra>'
            ))
    
    fig.update_layout(
        title='Bold Gel Ball - Rapunzel Variant Comparison',
        xaxis_title='Week',
        yaxis_title='Cumulative Repeat Rate (%)',
        hovermode='closest',
        height=500,
        showlegend=True,
        legend=dict(x=0.7, y=0.2)
    )
    
    fig.update_xaxes(range=[0, 14], gridcolor='lightgray')
    fig.update_yaxes(range=[0, 50], gridcolor='lightgray')
    
    fig.show()
    
    # Display reference table
    print("\n📊 Reference Table: Rapunzel Variant Weekly Data")
    pivot_rapunzel = rapunzel_weekly.pivot_table(
        index='week',
        columns='initiative_name',
        values='cumulative_repeat_rate',
        aggfunc='first'
    ).round(2)
    display(pivot_rapunzel)
else:
    print("⚠ No Rapunzel variant weekly data available")
    print("   Note: Variant-level data requires correct product name filters.")
    print("   Currently showing only 'Rapunzel (All)' - need to re-extract with variant patterns.")

⚠ No Rapunzel variant weekly data available
   Note: Variant-level data requires correct product name filters.
   Currently showing only 'Rapunzel (All)' - need to re-extract with variant patterns.


## 6. Size Migration Analysis

In [11]:
# Display size migration analysis (already calculated from cached data)
if len(all_size_df) > 0:
    print("\n" + "="*90)
    print("SIZE MIGRATION ANALYSIS - WHERE DO REPEAT SHOPPERS GO?")
    print("="*90)
    print("\nKey Question: Are shoppers returning to base/core product or trading up to larger sizes?")
    print("\n")
    
    # Show results grouped by initiative (focus on latest)
    latest_inits = ['Anakin (All)', 'Anakin (Fresh)', 'Anakin (Clean)', 'Anakin (BIO)',
                    'Rapunzel (All)', 'Rapunzel (Pink)', 'Rapunzel (Blue)', 'Rapunzel (WH-TEA&FL)']
    
    for initiative in latest_inits:
        if initiative in all_size_df['initiative_name'].values:
            print(f"\n--- {initiative} ---")
            initiative_data = all_size_df[all_size_df['initiative_name'] == initiative]
            display(initiative_data[['repeat_size', 'repeat_shoppers', 'repeat_rate_to_size', 'share_of_repeaters']])
else:
    print("⚠ No size migration data available.")


SIZE MIGRATION ANALYSIS - WHERE DO REPEAT SHOPPERS GO?

Key Question: Are shoppers returning to base/core product or trading up to larger sizes?



--- Anakin (All) ---


,repeat_size,repeat_shoppers,repeat_rate_to_size,share_of_repeaters
24,詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,37741,12.61,39.24
25,本体通常,24983,8.35,25.98
26,詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,12810,4.28,13.32
27,詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,8248,2.76,8.58
28,詰替ﾃﾗｼﾞｬﾝﾎﾞ,8037,2.69,8.36
29,詰替超ﾃﾗｼﾞｬﾝﾎﾞ,4338,1.45,4.51
30,ｿﾉﾀ,11,0.00,0.01
31,詰替超ｼﾞｬﾝﾎﾞ,1,0.00,0.00



--- Rapunzel (All) ---


,repeat_size,repeat_shoppers,repeat_rate_to_size,share_of_repeaters
32,詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,62992,18.44,40.71
33,本体通常,36378,10.65,23.51
34,詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,20441,5.99,13.21
35,詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,14725,4.31,9.52
36,詰替ﾃﾗｼﾞｬﾝﾎﾞ,13256,3.88,8.57
37,詰替超ﾃﾗｼﾞｬﾝﾎﾞ,6942,2.03,4.49
38,ｿﾉﾀ,10,0.00,0.01
39,詰替超ｼﾞｬﾝﾎﾞ,5,0.00,0.00


In [12]:
# Size migration pivot table for easier comparison
if len(all_size_df) > 0:
    # Create pivot: initiatives vs size segments
    latest_size = all_size_df[all_size_df['initiative_name'].isin(
        ['Anakin (All)', 'Anakin (Fresh)', 'Anakin (Clean)', 'Anakin (BIO)',
         'Rapunzel (All)', 'Rapunzel (Pink)', 'Rapunzel (Blue)', 'Rapunzel (WH-TEA&FL)']
    )]
    
    if len(latest_size) > 0:
        pivot_size = latest_size.pivot_table(
            index='repeat_size',
            columns='initiative_name',
            values='share_of_repeaters',
            aggfunc='first'
        ).round(1)
        
        print("\n" + "="*90)
        print("SIZE SHARE COMPARISON - % of Repeaters by Pack Size")
        print("="*90)
        display(pivot_size)


SIZE SHARE COMPARISON - % of Repeaters by Pack Size


initiative_name,Anakin (All),Rapunzel (All)
repeat_size,,
本体通常,26.0,23.5
詰替超ｼﾞｬﾝﾎﾞ,0.0,0.0
詰替超ﾃﾗｼﾞｬﾝﾎﾞ,4.5,4.5
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ,8.6,9.5
詰替ﾃﾗｼﾞｬﾝﾎﾞ,8.4,8.6
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ,39.2,40.7
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ,13.3,13.2
ｿﾉﾀ,0.0,0.0


## 7. Latest Initiative Variant Comparison (Anakin vs Rapunzel)

In [13]:
# =============================================================================
# Latest Initiative Variant Comparison
# Anakin (Ariel) Top 3 Variants vs Rapunzel (Bold) Variants
# =============================================================================

print("\n" + "="*90)
print("LATEST INITIATIVE VARIANT COMPARISON")
print("="*90)

# Anakin variants (Ariel Gel Ball - Latest)
anakin_variants = [r for r in overall_df.to_dict('records') if 'Anakin' in r['initiative_name']]
if anakin_variants:
    print("\n🔵 ARIEL GEL BALL - ANAKIN INITIATIVE (Nov-Dec 2025)")
    print("-" * 50)
    anakin_df = pd.DataFrame(anakin_variants)
    display_anakin = anakin_df.copy()
    display_anakin['pre_shoppers'] = display_anakin['pre_shoppers'].apply(lambda x: f"{x:,}")
    display_anakin['repeat_shoppers'] = display_anakin['repeat_shoppers'].apply(lambda x: f"{x:,}")
    display_anakin['repeat_rate'] = display_anakin['repeat_rate'].apply(lambda x: f"{x:.2f}%")
    display(display_anakin[['initiative_name', 'variant', 'pre_shoppers', 'repeat_shoppers', 'repeat_rate']])

# Rapunzel variants (Bold Gel Ball - Latest)
rapunzel_variants = [r for r in overall_df.to_dict('records') if 'Rapunzel' in r['initiative_name']]
if rapunzel_variants:
    print("\n🟣 BOLD GEL BALL - RAPUNZEL INITIATIVE (Oct-Dec 2025)")
    print("-" * 50)
    rapunzel_df = pd.DataFrame(rapunzel_variants)
    display_rapunzel = rapunzel_df.copy()
    display_rapunzel['pre_shoppers'] = display_rapunzel['pre_shoppers'].apply(lambda x: f"{x:,}")
    display_rapunzel['repeat_shoppers'] = display_rapunzel['repeat_shoppers'].apply(lambda x: f"{x:,}")
    display_rapunzel['repeat_rate'] = display_rapunzel['repeat_rate'].apply(lambda x: f"{x:.2f}%")
    display(display_rapunzel[['initiative_name', 'variant', 'pre_shoppers', 'repeat_shoppers', 'repeat_rate']])


LATEST INITIATIVE VARIANT COMPARISON

🔵 ARIEL GEL BALL - ANAKIN INITIATIVE (Nov-Dec 2025)
--------------------------------------------------


,initiative_name,variant,pre_shoppers,repeat_shoppers,repeat_rate
0,Anakin (All),all,"299,289","89,308",29.84%



🟣 BOLD GEL BALL - RAPUNZEL INITIATIVE (Oct-Dec 2025)
--------------------------------------------------


,initiative_name,variant,pre_shoppers,repeat_shoppers,repeat_rate
0,Rapunzel (All),all,"341,522","134,987",39.53%


## 8. Export Results

In [14]:
# Export all results to Excel
output_file = 'initiative_repeat_rate_analysis.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Overall results
    if len(overall_df) > 0:
        overall_df.to_excel(writer, sheet_name='Overall_Repeat_Rates', index=False)
    
    # Weekly dynamics
    if len(all_weekly_df) > 0:
        all_weekly_df.to_excel(writer, sheet_name='Weekly_Dynamics', index=False)
        if 'pivot_weekly' in dir():
            pivot_weekly.to_excel(writer, sheet_name='Weekly_Pivot')
    
    # Size migration
    if len(all_size_df) > 0:
        all_size_df.to_excel(writer, sheet_name='Size_Migration', index=False)
    
    # Raw transaction data (for validation)
    if len(all_txn_df) > 0:
        # Sample for debugging (full data may be too large)
        sample_txn = all_txn_df.sample(min(10000, len(all_txn_df)), random_state=42)
        sample_txn.to_excel(writer, sheet_name='Transaction_Sample', index=False)
    
    # Initiative parameters
    params_df = pd.DataFrame(initiatives)
    params_df.to_excel(writer, sheet_name='Parameters', index=False)
    
    # Metadata
    metadata_df = pd.DataFrame({
        'Item': ['Analysis Date', 'Category', 'Number of Initiatives', 'Number of Retailers', 
                 'Retailers', 'Total Transactions Cached', 'Extraction Time (seconds)'],
        'Value': [
            datetime.now().strftime('%Y-%m-%d'),
            category,
            str(len(initiatives)),
            str(len(customer_codes)),
            ', '.join(customer_codes),
            str(len(all_txn_df)),
            f"{extraction_time:.1f}" if 'extraction_time' in dir() else 'N/A'
        ]
    })
    metadata_df.to_excel(writer, sheet_name='Metadata', index=False)

print(f"\n✓ Results exported to: {output_file}")
print(f"  Sheets: Overall_Repeat_Rates, Weekly_Dynamics, Weekly_Pivot, Size_Migration, Transaction_Sample, Parameters, Metadata")


✓ Results exported to: initiative_repeat_rate_analysis.xlsx
  Sheets: Overall_Repeat_Rates, Weekly_Dynamics, Weekly_Pivot, Size_Migration, Transaction_Sample, Parameters, Metadata


## 9. Key Findings Summary

### ⚡ Performance Optimization
- **Previous**: 33 separate database queries (11 initiatives × 3 analyses each)
- **Optimized**: 2 consolidated queries (1 per brand) + pandas metric calculation
- **Expected Speedup**: 10-15x faster execution

### Analysis Methodology:
- **Pre-period Purchasers**: Shoppers who purchased the target initiative product during the pre-period
- **Repeat Shoppers**: Pre-period purchasers who bought the same sub-brand again AFTER their trial date (diff > 0)
- **Repeat Rate**: (Repeat Shoppers / Pre-period Purchasers) × 100%

### Initiatives Analyzed:
| Brand | Initiative | Variants |
|-------|------------|----------|
| Ariel Gel Ball | Srixon, Srixon Boost, Yoda | All |
| Ariel Gel Ball | **Anakin (Latest)** | **All, Fresh, Clean, BIO** |
| Bold Gel Ball | Moana, Cinderella, Snowwhite | All |
| Bold Gel Ball | **Rapunzel (Latest)** | **All, Pink, Blue, WH-TEA&FL** |

### Key Questions Answered:
1. Which initiative performed best in terms of repeat rate?
2. Which variant within Anakin (Ariel) performed best?
3. Which variant within Rapunzel (Bold) performed best?
4. How does initial repeat momentum compare across variants?
5. Are repeat shoppers trading up to larger sizes or returning to base products?